# 9.5 - LoRA & QLoRA Adapters

**Module 9 - Fine-Tuning** | Netsetos GenAI Engineering

This notebook works through LoRA & QLoRA Adapters hands-on: LoRA from scratch, the whole trick in ~15 lines; The parameter math, no torch required; The α/r scaling trap; NF4 quantization and QLoRA memory; What to fix first: LR > targets > rank; Which LoRA variant to switch on.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

One commented `pip install` line for the whole lesson. Every code cell below runs on either the standard library or `torch` alone — the heavier stack (transformers, trl, peft, bitsandbytes) is only needed for the illustrative PEFT pipeline, which is shown as text, not executed. Uncomment and run this once on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install transformers torch "trl>=1.0" peft transformers datasets peft torch python-dotenv -q  # noqa


**Explanation**

Environment prep: a single install line, commented out so it never fires by accident on a machine that already has the packages.

**How the code works - step by step**
- **`transformers` / `torch`** — the model and tensor library; `torch` is the only dependency the runnable cells actually touch.
- **`trl>=1.0`** — the training loop (SFTTrainer/SFTConfig) referenced in the illustrative pipeline; the `>=1.0` pin matters because v1.0 renamed several arguments.
- **`peft`** — the adapter library (LoraConfig, DoRA/rsLoRA flags) shown in the pipeline.
- **`datasets`** — dataset loading for a real training run.
- **`python-dotenv`** — lets the next cell pull keys from a `.env` file.

**In one line:** uncomment once to provision the full fine-tuning stack; the runnable demos need only torch.

**What you'll see:** (no output - environment prep)

## Setup - environment keys

A safety-first key loader. Nothing in this notebook actually calls a hosted model — every cell is a local calculator or a torch demo — so these slots stay empty and unused here. The pattern is carried over from the API lessons so the notebook is ready if you extend it.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A no-op-by-default credential block: `setdefault` only fills a variable that is not already set, so a real key in your environment is never overwritten and an empty default is harmless because no cell here makes a network call.

**How the code works - step by step**
- **`import os`** — access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** and the ANTHROPIC / GOOGLE lines — register each key name with an empty fallback, leaving any real value already in the environment untouched.
- The comments point at where each key is issued (platform.openai.com, console.anthropic.com, aistudio.google.com).

**In one line:** declare the key slots without hardcoding secrets; none are exercised by the code that follows.

**What you'll see:** (no output - environment prep)

## 1 - LoRA from scratch, the whole trick in ~15 lines

LoRA has exactly one idea: freeze the pretrained weight matrix and bolt on two skinny matrices whose product is your entire fine-tune. Building the layer yourself is the fastest way to see why 0.39% of the parameters can match a full fine-tune — and why the adapter starts life doing literally nothing.

> **Needs torch** to run; the parameter counts are also derivable by hand (next cell).

In [ ]:
# LoRA FROM SCRATCH - the whole mechanism in ~15 lines (runnable with torch).
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    """Wrap a frozen linear layer: output = W0(x) + scale * (x @ A @ B)."""
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad = False        # FREEZE the pretrained weights
        if self.base.bias is not None:
            self.base.bias.requires_grad = False      # freeze the bias too (nn.Linear defaults to bias=True)
        d_in, d_out = base.in_features, base.out_features
        self.A = nn.Parameter(torch.randn(d_in, rank) * 0.01)   # small random
        self.B = nn.Parameter(torch.zeros(rank, d_out))         # ZERO -> adapter starts as a no-op
        self.scale = alpha / rank                      # this scalar is the whole "alpha/r" story

    def forward(self, x):
        return self.base(x) + (x @ self.A @ self.B) * self.scale

d = 4096
lora = LoRALayer(nn.Linear(d, d), rank=8, alpha=16)
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in lora.parameters() if not p.requires_grad)
print(f"frozen W0: {frozen:,}  |  trainable A+B: {trainable:,}  ({trainable/frozen*100:.2f}%)")

# Output: (runnable with torch installed)
# frozen W0: 16,781,312  |  trainable A+B: 65,536  (0.39%)

**Explanation**

A minimal `nn.Module` that wraps any frozen `nn.Linear` and adds a trainable low-rank detour. The whole model is: keep W0 fixed, push the input through a squeeze-down matrix A and an expand-back matrix B, scale the result, and add it to the frozen output. Because B is all zeros at initialization, the wrapped layer is bit-identical to the base at step 0 and only diverges as training moves A and B.

**How the code works - step by step**
- **`__init__`** — stores the frozen `base` layer and sets `base.weight.requires_grad = False` (and the bias too) so no gradient ever flows into the pretrained matrix.
- **`self.A`** — a `d_in × rank` matrix of small random values (`* 0.01`): the on-ramp that squeezes the input to rank `r`.
- **`self.B`** — a `rank × d_out` matrix of **zeros**: the off-ramp back to full width, initialized so the detour outputs 0 and the adapter begins as a no-op.
- **`self.scale = alpha / rank`** — the single scalar that is the entire α/r story explored in Concept 3.
- **`forward`** — returns `base(x) + (x @ A @ B) * scale`: the frozen path plus the trained detour.
- The driver builds a 4096×4096 layer with `rank=8, alpha=16` and counts trainable (A+B) vs frozen (W0) parameters.

**In one line:** freeze W0, add a zero-initialized B·A detour scaled by α/r, and train only the detour.

**What you'll see:** Prints `frozen W0: 16,781,312 | trainable A+B: 65,536 (0.39%)` — two small matrices carry the fine-tune while 99.6% of the layer stays frozen.

## 2 - The parameter math, no torch required

The savings are just arithmetic. A full layer has `d_in × d_out` weights; LoRA replaces the update with `r × (d_in + d_out)`. This cell tabulates the trade-off across ranks and scales it up to a whole 7B model so you can see where the ~59MB adapter figure comes from.

In [ ]:
# LoRA PARAMETER MATH - no torch needed. A is r x d_in, B is d_out x r.
def lora_params(d_in, d_out, r):
    return r * (d_in + d_out)          # A: r*d_in  +  B: d_out*r
def full_params(d_in, d_out):
    return d_in * d_out

d = 4096
for r in (4, 8, 16, 64):
    lp, fp = lora_params(d, d, r), full_params(d, d)
    print(f"  r={r:<3} one {d}x{d} layer: {lp:>8,} trainable vs {fp:,} full  ({lp/fp*100:.2f}%)")

whole = lora_params(d, d, 16) * 7 * 32   # ~7 all-linear modules x ~32 layers
print(f"  r=16 all-linear across a 7B: ~{whole/1e6:.1f}M trainable (~{whole/7e9*100:.2f}% of 7B) -> a ~{whole*2/1e6:.0f}MB adapter (bf16)")

# Output:
#   r=4   one 4096x4096 layer:   32,768 trainable vs 16,777,216 full  (0.20%)
#   r=8   one 4096x4096 layer:   65,536 trainable vs 16,777,216 full  (0.39%)
#   r=16  one 4096x4096 layer:  131,072 trainable vs 16,777,216 full  (0.78%)
#   r=64  one 4096x4096 layer:  524,288 trainable vs 16,777,216 full  (3.12%)
#   r=16 all-linear across a 7B: ~29.4M trainable (~0.42% of 7B) -> a ~59MB adapter (bf16)

**Explanation**

A pure-arithmetic harness — no model, no torch — that contrasts LoRA's parameter count against a full layer and extrapolates to an all-linear 7B fine-tune. It exists to make the compression ratio concrete and rank-by-rank.

**How the code works - step by step**
- **`lora_params(d_in, d_out, r)`** — returns `r * (d_in + d_out)`, the size of A (`r·d_in`) plus B (`d_out·r`).
- **`full_params(d_in, d_out)`** — returns `d_in * d_out`, the full dense update for comparison.
- The loop over `r in (4, 8, 16, 64)` prints trainable-vs-full and the percentage for one 4096×4096 layer.
- **`whole = lora_params(...) * 7 * 32`** — approximates rank-16 adapters across ~7 linear modules over ~32 layers of a 7B model, then reports the trainable count, its share of 7B, and the resulting bf16 adapter size (`*2` bytes).

**In one line:** trainable params grow linearly with rank while the full-layer cost is fixed, so even rank-64 is a few percent.

**What you'll see:** A table: r=4 → 0.20%, r=8 → 0.39%, r=16 → 0.78%, r=64 → 3.12% of one layer, then the summary line `r=16 all-linear across a 7B: ~29.4M trainable (~0.42% of 7B) -> a ~59MB adapter (bf16)`.

## 3 - The α/r scaling trap

Every LoRA forward multiplies the detour by `scale = α/r`. Hold α fixed and raise the rank — as many tutorials do — and that scale quietly shrinks, weakening the effective learning rate the adapter feels. This is the real reason behind "I raised the rank and nothing improved."

In [ ]:
# THE alpha/r SCALING - and why "raising rank did nothing".
import math
def scale_lora(alpha, r):   return alpha / r
def scale_rslora(alpha, r): return alpha / math.sqrt(r)

print("  alpha = 2*r (Raschka convention):")
print(f"  {'rank':>5} {'alpha':>6} {'LoRA a/r':>10} {'rsLoRA a/sqrt(r)':>18}")
for r in (8, 16, 64, 256):
    a = 2 * r
    print(f"  {r:>5} {a:>6} {scale_lora(a, r):>10.3f} {scale_rslora(a, r):>18.3f}")

# Now hold alpha FIXED and watch the effective update shrink as rank grows:
a = 16
s16, s256 = scale_lora(a, 16), scale_lora(a, 256)
print(f"  With FIXED alpha={a}: scale at r=16 is {s16:.4f}, at r=256 is {s256:.4f} -> {s16/s256:.0f}x smaller.")
print(f"  That silent {s16/s256:.0f}x shrink IS why 'raising rank did nothing' - re-tune LR or use rsLoRA.")

# Output:
#    rank  alpha   LoRA a/r   rsLoRA a/sqrt(r)
#       8     16      2.000              5.657
#      16     32      2.000              8.000
#      64    128      2.000             16.000
#     256    512      2.000             32.000
#   With FIXED alpha=16: scale at r=16 is 1.0000, at r=256 is 0.0625 -> 16x smaller.
#   That silent 16x shrink IS why 'raising rank did nothing' - re-tune LR or use rsLoRA.

**Explanation**

A tiny numeric demonstration of how the adapter's scale behaves under two conventions. It contrasts the standard α=2r rule (which pins the scale at a constant) against a fixed-α mistake (which lets the scale collapse), and shows rsLoRA's √r fix. It is a measurement, not a training run.

**How the code works - step by step**
- **`scale_lora(alpha, r)`** — returns `alpha / r`, the vanilla LoRA scale.
- **`scale_rslora(alpha, r)`** — returns `alpha / sqrt(r)`, the rsLoRA variant.
- The first loop uses the `alpha = 2*r` convention and prints both scales for ranks 8→256, showing LoRA stays flat at 2.0 while rsLoRA climbs.
- The second block **holds α fixed at 16** and computes the scale at r=16 vs r=256, then the ratio — the hidden shrink.

**In one line:** with fixed α, going to higher rank silently divides your effective update; re-tune LR or use rsLoRA.

**What you'll see:** The α=2r table shows LoRA pinned at 2.000 while rsLoRA rises 5.657 → 32.000, then: `With FIXED alpha=16: scale at r=16 is 1.0000, at r=256 is 0.0625 -> 16x smaller`, flagged as why raising rank did nothing.

## 4 - NF4 quantization and QLoRA memory

LoRA shrinks what you train; QLoRA shrinks what you store, loading the frozen base in 4-bit. The clever part is the number format: NF4 places its 16 levels at the quantiles of a normal distribution, spending precision near zero where trained weights actually cluster. This cell reconstructs NF4's levels and measures its error against uniform INT4.

In [ ]:
# NF4 vs INT4, and QLoRA memory. NF4 puts its 16 levels at N(0,1) quantiles.
import math
def norm_ppf(p):   # inverse normal CDF (Acklam) - enough for a demo
    a=[-3.969683028665376e+01,2.209460984245205e+02,-2.759285104469687e+02,1.383577518672690e+02,-3.066479806614716e+01,2.506628277459239e+00]
    b=[-5.447609879822406e+01,1.615858368580409e+02,-1.556989798598866e+02,6.680131188771972e+01,-1.328068155288572e+01]
    c=[-7.784894002430293e-03,-3.223964580411365e-01,-2.400758277161838e+00,-2.549732539343734e+00,4.374664141464968e+00,2.938163982698783e+00]
    dd=[7.784695709041462e-03,3.224671290700398e-01,2.445134137142996e+00,3.754408661907416e+00]
    if p < 0.02425:
        q=math.sqrt(-2*math.log(p));  return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((dd[0]*q+dd[1])*q+dd[2])*q+dd[3])*q+1)
    if p <= 0.97575:
        q=p-0.5; r=q*q;               return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q/(((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)
    q=math.sqrt(-2*math.log(1-p));    return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((dd[0]*q+dd[1])*q+dd[2])*q+dd[3])*q+1)

raw = [norm_ppf((i + 0.5) / 16) for i in range(16)]
mx = max(abs(v) for v in raw)
nf4  = [round(v / mx, 3) for v in raw]                 # 16 quantile levels, normalized
int4 = [round(-1 + 2*i/15, 3) for i in range(16)]      # 16 UNIFORM levels
near = lambda levels, x: min(levels, key=lambda l: abs(l - x))
weights = [-0.9,-0.4,-0.2,-0.1,-0.05,-0.02,0.0,0.03,0.06,0.1,0.15,0.3,0.5,0.8]   # clustered near 0
e_nf4  = sum(abs(w - near(nf4,  w)) for w in weights) / len(weights)
e_int4 = sum(abs(w - near(int4, w)) for w in weights) / len(weights)
print(f"  NF4 levels near 0: {sorted(l for l in nf4 if abs(l) < 0.2)}")
print(f"  mean abs quant error (near-zero weights):  NF4={e_nf4:.4f}   INT4={e_int4:.4f}  -> NF4 {e_int4/e_nf4:.2f}x lower")
P = 7e9
print(f"  7B base: fp16 ~{P*2/1e9:.1f}GB -> 4-bit NF4 ~{P*0.5/1e9:.1f}GB; double-quant saves ~{P*(0.37/8)/1e9:.1f}GB more -> fits a 16GB T4")

# Output:
#   NF4 levels near 0: [-0.127, -0.042, 0.042, 0.127]
#   mean abs quant error (near-zero weights):  NF4=0.0326   INT4=0.0374  -> NF4 1.15x lower
#   7B base: fp16 ~14.0GB -> 4-bit NF4 ~3.5GB; double-quant saves ~0.3GB more -> fits a 16GB T4

**Explanation**

A from-scratch NF4-vs-INT4 comparison plus a QLoRA memory estimate. It rebuilds the 16 NF4 levels from the inverse normal CDF, quantizes a near-zero weight sample with both formats, and computes each one's mean error — a measurement harness that shows *why* NF4 wins on weights, not just that it does.

**How the code works - step by step**
- **`norm_ppf(p)`** — an Acklam-approximation inverse normal CDF, used to find where the quantile levels fall.
- **`raw`/`nf4`** — 16 quantile points at `(i+0.5)/16`, normalized by the max magnitude: NF4's non-uniform levels, crowded near zero.
- **`int4`** — 16 evenly spaced levels from -1 to 1: the uniform baseline.
- **`near(levels, x)`** — snaps a weight to its closest level; applied to a hand-picked `weights` list clustered near zero.
- **`e_nf4` / `e_int4`** — mean absolute quantization error for each format, then their ratio.
- The final lines estimate a 7B base at fp16 (~14GB) vs 4-bit NF4 (~3.5GB) and the extra double-quant saving.

**In one line:** put the ticks where the weights are (near zero) and the same 4 bits measure them more accurately, while the base drops to T4-sized memory.

**What you'll see:** Prints the near-zero NF4 levels (±0.042, ±0.127), `NF4=0.0326 INT4=0.0374 -> NF4 1.15x lower` error, and the memory line: fp16 ~14.0GB → NF4 ~3.5GB, double-quant ~0.3GB more, fits a 16GB T4.

## 5 - What to fix first: LR > targets > rank

The most useful result in the recent LoRA literature is a ranking of the knobs. Sweep the learning rate first (LoRA's optimum is ~10× full fine-tuning's), target all-linear layers second, and stop obsessing over rank. This cell encodes that hierarchy as a diagnostic that always surfaces the highest-priority fix.

In [ ]:
# WHAT TO FIX FIRST - the 2024-2026 hierarchy encoded: LR > targets > rank.
def diagnose(lr_retuned, targets, rank):
    fixes = []
    if not lr_retuned:
        fixes.append(("LR", "re-tune LR: LoRA's optimum is ~10x full-FT; a rank change shifts the effective LR"))
    if targets == "attention":
        fixes.append(("TARGETS", "use all-linear: MLP matters more than attention; attention-only stunts learning"))
    if rank < 8:
        fixes.append(("RANK", "raise rank to >=8 (rank is the LEAST important knob, but <8 can underfit)"))
    if not fixes:
        fixes.append(("OK", "LR retuned + all-linear + sane rank -> on the frontier; improve DATA next"))
    return fixes

for cfg in [(False, "attention", 4), (False, "all-linear", 16), (True, "attention", 64), (True, "all-linear", 16)]:
    fixes = diagnose(*cfg)
    print(f"  lr_retuned={str(cfg[0]):5} targets={cfg[1]:<10} rank={cfg[2]:<3} -> FIX {fixes[0][0]}: {fixes[0][1]}")

# Output:
#   lr_retuned=False targets=attention  rank=4   -> FIX LR: re-tune LR: LoRA's optimum is ~10x full-FT; a rank change shifts the effective LR
#   lr_retuned=False targets=all-linear rank=16  -> FIX LR: re-tune LR: LoRA's optimum is ~10x full-FT; a rank change shifts the effective LR
#   lr_retuned=True  targets=attention  rank=64  -> FIX TARGETS: use all-linear: MLP matters more than attention; attention-only stunts learning
#   lr_retuned=True  targets=all-linear rank=16  -> FIX OK: LR retuned + all-linear + sane rank -> on the frontier; improve DATA next

**Explanation**

A rule-based diagnoser, not a model — it takes a config (was the LR retuned, which modules, what rank) and returns the single most impactful change, ordered by the 2024-2026 ablation hierarchy. It turns a research finding into a checklist you can run against any underperforming LoRA run.

**How the code works - step by step**
- **`diagnose(lr_retuned, targets, rank)`** — appends fixes in priority order: an un-retuned LR first, attention-only targets second, a rank below 8 third; if none apply it recommends improving the data.
- The priority order is baked into the append sequence, so `fixes[0]` is always the top lever.
- The loop runs four representative configs and prints only the highest-priority fix for each.

**In one line:** LR beats target-coverage beats rank — the code enforces that ordering so you fix the biggest lever first.

**What you'll see:** Four lines: the two un-retuned-LR configs both flag `FIX LR` (even the all-linear one), the retuned-but-attention-only config flags `FIX TARGETS`, and only the retuned + all-linear + sane-rank config reports `FIX OK: ... improve DATA next`.

## 6 - Which LoRA variant to switch on

Beyond vanilla LoRA, each variant changes one thing and each is a single PEFT flag: DoRA (magnitude + direction, the 2026 default), rsLoRA (the α/√r fix), PiSSA (SVD init, faster SFT — but it fails in RL), VeRA (shared frozen matrices for extreme parameter thrift). This cell picks the right flags for your goal, rank, and budget.

In [ ]:
# WHICH VARIANT? - DoRA / rsLoRA / PiSSA / VeRA as PEFT flags.
def pick_variant(goal, rank, param_budget):
    flags = {"use_dora": True}
    notes = ["use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)"]
    if rank >= 32:
        flags["use_rslora"] = True
        notes.append("use_rslora=True (alpha/sqrt(r) - stable gradients at high rank)")
    if goal == "sft-fast-converge":
        notes.append('init_lora_weights="pissa" (SVD init; faster convergence for SFT)')
    if goal == "rl":
        flags = {"use_dora": False}
        notes = ["vanilla LoRA for RL: PiSSA FAILS in GRPO; rank=1 can suffice (LoRA Without Regret)"]
    if param_budget == "extreme-low":
        notes.append("consider VeRA (shared frozen A,B; ~10x fewer trainable params)")
    return flags, notes

for goal, rank, budget in [("sft",16,"normal"),("sft-fast-converge",64,"normal"),("rl",8,"normal"),("sft",16,"extreme-low")]:
    flags, notes = pick_variant(goal, rank, budget)
    print(f"  goal={goal:18s} rank={rank:<3} budget={budget:12s} -> {flags}")
    for n in notes:
        print(f"      - {n}")

# Output:
#   goal=sft                rank=16  budget=normal       -> {'use_dora': True}
#       - use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)
#   goal=sft-fast-converge  rank=64  budget=normal       -> {'use_dora': True, 'use_rslora': True}
#       - use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)
#       - use_rslora=True (alpha/sqrt(r) - stable gradients at high rank)
#       - init_lora_weights="pissa" (SVD init; faster convergence for SFT)
#   goal=rl                 rank=8   budget=normal       -> {'use_dora': False}
#       - vanilla LoRA for RL: PiSSA FAILS in GRPO; rank=1 can suffice (LoRA Without Regret)
#   goal=sft                rank=16  budget=extreme-low  -> {'use_dora': True}
#       - use_dora=True (magnitude+direction; ~+1-4%, merges to zero inference cost)
#       - consider VeRA (shared frozen A,B; ~10x fewer trainable params)

**Explanation**

A recommender that maps a training scenario to concrete PEFT config flags. It layers the variant rules — DoRA on by default, rsLoRA above rank 32, PiSSA for fast SFT convergence, and a hard override for RL where PiSSA is known to fail — so you can read the reasoning, not just the answer.

**How the code works - step by step**
- **`pick_variant(goal, rank, param_budget)`** — starts with `use_dora=True` and a note.
- **`if rank >= 32`** — adds `use_rslora=True` for stable gradients at high rank.
- **`if goal == "sft-fast-converge"`** — suggests PiSSA's SVD init.
- **`if goal == "rl"`** — overrides to vanilla LoRA (`use_dora=False`) because PiSSA fails in GRPO and rank-1 can suffice.
- **`if param_budget == "extreme-low"`** — recommends VeRA's shared frozen A,B.
- The loop prints the chosen flags and rationale for four scenarios.

**In one line:** DoRA by default, add rsLoRA at high rank, PiSSA for fast SFT, drop to vanilla LoRA for RL — all one-flag changes.

**What you'll see:** Four scenarios with their flag dicts: SFT r=16 → `{use_dora: True}`; sft-fast-converge r=64 → DoRA + rsLoRA + PiSSA note; RL → `{use_dora: False}` with the PiSSA-fails warning; extreme-low budget → DoRA plus a VeRA suggestion.

## 7 - Merge vs multi-LoRA serving

A trained adapter is a tiny file, and one base model can serve many of them. Merge (`merge_and_unload`) folds B·A back into W0 for a single task with zero overhead; keep adapters separate and one base plus N tiny adapters serves many tasks, routed per request. This cell computes the memory crossover.

In [ ]:
# MERGE vs MULTI-LoRA - the serving memory tradeoff.
def lora_params(d_in, d_out, r): return r * (d_in + d_out)
def base_gb(B):           return B * 2.0                          # fp16 base
def adapter_gb(B, r):     return lora_params(4096,4096,r)*7*32*(B/7.0)*2/1e9
def naive(n, B):          return n * base_gb(B)                   # N merged copies
def multilora(n, B, r):   return base_gb(B) + n * adapter_gb(B, r)  # 1 base + N adapters

B, r = 7, 16
print("  7B base, r=16 adapters. Serving N task adapters:")
print(f"  {'N':>3} {'naive (N copies)':>18} {'multi-LoRA':>14} {'saved':>10}")
for n in (1, 3, 5, 10):
    nv, ml = naive(n, B), multilora(n, B, r)
    print(f"  {n:>3} {nv:>16.1f}GB {ml:>12.1f}GB {nv-ml:>8.1f}GB")
print(f"  one r=16 adapter ~{adapter_gb(B,r)*1000:.0f}MB vs a {base_gb(B):.0f}GB base.")
print(f"  Merge (merge_and_unload) only for a SINGLE task: zero overhead, but you lose per-task routing.")

# Output:
#   7B base, r=16 adapters. Serving N task adapters:
#     N   naive (N copies)     multi-LoRA      saved
#     1             14.0GB         14.1GB     -0.1GB
#     3             42.0GB         14.2GB     27.8GB
#     5             70.0GB         14.3GB     55.7GB
#    10            140.0GB         14.6GB    125.4GB
#   one r=16 adapter ~59MB vs a 14GB base.
#   Merge (merge_and_unload) only for a SINGLE task: zero overhead, but you lose per-task routing.

**Explanation**

A serving-memory calculator that contrasts the naive approach (N full merged model copies) against multi-LoRA (one base plus N small adapters). It reuses the same `lora_params` arithmetic from Concept 2 to size the adapters and shows how the saving explodes with task count — a deployment decision made numeric.

**How the code works - step by step**
- **`base_gb(B)`** — a B-billion-param fp16 base at 2 bytes/param.
- **`adapter_gb(B, r)`** — one all-linear rank-r adapter's size, scaled from the 4096-dim per-layer figure.
- **`naive(n, B)`** — N independent merged copies: `n * base_gb`.
- **`multilora(n, B, r)`** — one base plus N adapters: `base_gb + n * adapter_gb`.
- The loop prints naive vs multi-LoRA vs saving for N = 1, 3, 5, 10, then the per-adapter-vs-base size and the merge caveat.

**In one line:** past one task, one base + N small adapters crushes N full copies — merge only when you serve a single task.

**What you'll see:** A table where the two paths tie at N=1 (~14GB) but diverge fast: N=5 is 70.0GB naive vs 14.3GB multi-LoRA (55.7GB saved), N=10 saves 125.4GB; one r=16 adapter is ~59MB against a 14GB base.

## 8 - India LoRA patterns: Airavata & multilingual adapters

The headline Indic-fine-tuning result is a LoRA argument: LoRA ADDS a language like Hindi while preserving English, whereas full fine-tuning on Hindi data degrades English. And for several languages, separate language-specific adapters beat one joint adapter, because a joint run lets high-resource languages dominate the gradient. This cell turns that into a planner.

In [ ]:
# INDIA LoRA PLANNER - one joint adapter or many? Airavata's lesson: LoRA preserves English.
def india_plan(n_langs, resource):
    if n_langs == 1:
        return ("single LoRA (Airavata recipe: r=16, alpha=32, all proj, LR 5e-4, 4 epochs)",
                "adds the language while PRESERVING English (full FT on Hindi DEGRADES English)")
    if resource == "low":
        return (f"{n_langs} language-specific LoRA adapters + vLLM multi-LoRA (route by detected language)",
                "language-specific beats a joint adapter: high-resource langs dominate a joint gradient")
    return (f"{n_langs} adapters; evaluate joint vs separate, serve via multi-LoRA",
            "start separate; merge only if the languages are close and you want one model")

for n, res in [(1, "low"), (5, "low"), (2, "high")]:
    stack, why = india_plan(n, res)
    print(f"  langs={n} resource={res:5s} -> {stack}")
    print(f"      why: {why}")
print("  Memory: 1 base (~14GB) + 5 adapters (~295MB) << 5 merged models (~70GB).")

# Output:
#   langs=1 resource=low   -> single LoRA (Airavata recipe: r=16, alpha=32, all proj, LR 5e-4, 4 epochs)
#       why: adds the language while PRESERVING English (full FT on Hindi DEGRADES English)
#   langs=5 resource=low   -> 5 language-specific LoRA adapters + vLLM multi-LoRA (route by detected language)
#       why: language-specific beats a joint adapter: high-resource langs dominate a joint gradient
#   langs=2 resource=high  -> 2 adapters; evaluate joint vs separate, serve via multi-LoRA
#       why: start separate; merge only if the languages are close and you want one model
#   Memory: 1 base (~14GB) + 5 adapters (~295MB) << 5 merged models (~70GB).

**Explanation**

A decision helper that recommends a multilingual adapter strategy from the language count and resource level, encoding the Airavata recipe and the separate-beats-joint finding. Like the earlier planners it returns both the recommendation and the reasoning, ending with the memory comparison that makes the case.

**How the code works - step by step**
- **`india_plan(n_langs, resource)`** — for one language, returns the Airavata single-LoRA recipe (r=16, α=32, all projections, LR 5e-4, 4 epochs) with the English-preservation note.
- For multiple low-resource languages, recommends N language-specific adapters plus vLLM multi-LoRA routing, because a joint gradient is dominated by high-resource languages.
- Otherwise (higher-resource, few languages) suggests starting separate and evaluating joint vs separate.
- The loop prints the plan and rationale for three cases, then the memory line.

**In one line:** one adapter to add one language and preserve English; separate adapters + multi-LoRA routing when several low-resource languages compete.

**What you'll see:** Three plans: 1 language → the Airavata single-LoRA recipe (preserves English); 5 low-resource → 5 language-specific adapters + vLLM routing; 2 high-resource → start separate and evaluate. Closes with `1 base (~14GB) + 5 adapters (~295MB) << 5 merged models (~70GB)`.

Every cell here is a pure-Python or torch measurement of one LoRA idea: the frozen base plus a low-rank B·A detour (Concept 1), the parameter arithmetic that makes it cheap (Concept 2), the α/r scale that silently rescales your learning rate (Concept 3), NF4 quantization that shrinks the frozen base to 4-bit (Concept 4), and then the decision logic — what to fix first, which variant to flip on, whether to merge or serve many adapters, and how to stack it for multilingual work (Concepts 5-8). None of them call a model API; they are calculators and diagnostics that turn the lesson's claims into numbers you can re-run. From here, Lesson 9.6 (Evaluation, Merging & Deployment) takes the adapter you now understand through eval harnesses, TIES/DARE merging in depth, and production serving — where the merge-vs-multi-LoRA math above becomes hands-on.